# How far apart are the atoms in iron? (MSc)

**Your mission.** You are a method scientist in the Neutron Solutions Team of ESS. Fyrø Energy Island receives steel
plates for the foundations of its wind turbines, sold as *ferritic* steel (α-iron, body-centered cubic); a mix-up with
*austenitic* steel (γ-iron, face-centered cubic: not magnetic, expands more when heated) would matter, and the plates
may not be cut. Fyrø will check every delivery: it needs a method it can trust and repeat. The Bragg edges of the
neutron transmission reveal the crystal structure through centimeters of steel. You measure a first plate with ODIN,
develop the analysis, and give your verdict with its uncertainty (end of part 6).



*What you have to decide: α-iron (left) or γ-iron (middle)? Each structure has its own atomic planes, hence its own Bragg edges (right).*

You will measure the **Bragg edges** of an iron plate with the virtual ODIN and deduce the spacing of its atomic planes.
Parts 1 to 5 are guided: they take your files from the raw neutrons to the transmission $T(\lambda)$, with the
official ESS reduction. Part 6 is yours: **you develop the analysis with a model builder** (choose the model, the
range and the start values, fit, compare; no code to write), and a check cell tells you whether your result is right
(the twin knows the true sample). Part 7 is about the method itself: a systematic error to track down.

You start by measuring, in NICOS (part 0), as a user of ESS would.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))     # the modules of the ODIN twin (folder odin/)

**Your sample** (from the sample library):

In [ ]:
sys.path.insert(0, os.path.abspath("../../.."))  # the sample library (twin/sample_library.py)
import sample_library as sl, matplotlib.pyplot as plt
for name, hidden in sl.exercise_hidden("exercise.yml").items():
    spec = sl.load("odin")[name]
    print(sl.describe("odin", spec, hidden))
    sl.draw("odin", spec, hidden=hidden, fig=plt.figure(figsize=(10, 3.5))); sl.show()

In [ ]:
import quiz, odin_exercises as ex
q = quiz.Quiz("odin_iron_msc")       # the questions of this notebook

## Before you measure: three measurements, and why

A transmission is a ratio: what crosses the sample divided by what arrives without it. ODIN only counts neutrons, so you
need **three measurements**, with the same chopper settings and the same pinhole:

1. **the dark**: shutter closed, no beam at all. The camera still records a few counts (gamma rays, electronic noise):
   this background is in every measurement and has to be subtracted from the two others;
2. **the open beam**: shutter open, sample moved out of the beam. It tells how many neutrons of each wavelength arrive
   *without* the sample. The camera sees only 14 mm × 14 mm, so moving the plate 500 mm aside is more than enough;
3. **the sample**: shutter open, the iron plate in the beam.

The transmission is then $T(\lambda) = \dfrac{\text{sample} - \text{dark}}{\text{open beam} - \text{dark}}$, wavelength by
wavelength (part 5 adds the corrections).

In [ ]:
q.choice("Why is the open beam measured with exactly the same chopper settings and pinhole as the sample?",
         ["Because the transmission divides the two: they must see the same beam, so that only the sample makes the "
          "difference.",
          "To save time: the instrument does not have to move.",
          "Because NICOS refuses other settings.",
          "It does not matter: the reduction corrects any change of settings."],
         answer=0, why={1: "Moving the choppers takes a few seconds; the reason is the physics of the ratio.",
                        2: "NICOS would accept them; the ratio would then mix two different beams.",
                        3: "The reduction cannot correct a different beam: it only divides."},
         explanation="The ratio sample / open beam removes everything that is the same in both (the shape of the beam, "
                     "the efficiency of the camera). A different setting changes the beam itself, and the ratio would "
                     "no longer be the transmission of the sample.",
         hints=["What does a ratio cancel?"], title="Q1. Same settings")

## 0. Your measurements

As at ESS, the measurements are yours: nobody measures for you, and if a file is wrong (shutter forgotten, sample in
the wrong place), you will see it in part 2 and measure again. Choose how:

- **in NICOS**, the control software of the instrument at ESS: you type commands, one at a time, each explained below.
  This is how you will work at a real beamtime;
- **here, with a panel**: the same devices, set with buttons, and a *Count* button (as on the BSc page).

Either way your files go to the folder of proposal 1234, and parts 1 to 7 read them. You can change at any time.

In [ ]:
import measure_panel
measure_panel.Panel("odin", proposal="1234", samples=["iron_plate"])

<span class="twin-nicos"></span>

### Step 1. Your proposal

At ESS nothing is recorded without an accepted **proposal**: it gives the number of the experiment, the team and the
samples, and the files go to its folder. The proposal of this exercise is **1234**.

In NICOS, tab **Experiment**, click **Find my proposal**. NICOS asks the user office for your proposals. If you have
only one, NICOS fills in the form at once (number 1234, title, users, sample); if you have several, it lets you choose.
Check that the number is 1234, then click **Apply**: the proposal becomes the current experiment. Nothing is mounted
yet: NICOS lists the sample of the proposal, the iron plate, and mounting it is your first step (below).

Then go to the tab **Instrument interaction**. Type each command below in the command line at the top, then press
*Enter*, one at a time; the *Output* tab shows what NICOS does.

<span class="twin-nicos"></span>

### Step 2. The instrument

```
move(sample_holder, 'iron_plate')
sample_info()
```
mounts the sample of your proposal on the sample changer, then describes it: an iron plate of 10 mm × 10 mm, 5 mm
thick. Check it before measuring.

```
move(choppers, 'wfm')
```
sets the choppers to **WFM** mode (*wavelength frame multiplication*): the wavelength of each neutron is then known
to better than 1 %, which is needed to see the sharp jumps of a Bragg edge.

```
move(wfm_delta, 0.3)
```
sets the distance between the two WFM discs to 0.3 m: a good resolution with enough neutrons (a smaller distance gives
a finer resolution, but fewer neutrons).

```
move(pinhole, 0.03)
```
chooses the pinhole of 3 cm: a sharp enough image of the plate, with many neutrons.

<span class="twin-nicos"></span>

### Step 3. The dark

```
move(shutter, 'closed')
```
closes the shutter just before the sample: no neutron reaches the camera.

```
with nexusfile_open('dark'): count(t=30)
```
counts for 30 s and writes **one file**, titled *dark*. `nexusfile_open` is the ESS command that records everything
measured inside it in one NeXus file, in the folder of your proposal. The *Detector image* stays almost empty: only the
background.

<span class="twin-nicos"></span>

### Step 4. The open beam

```
move(shutter, 'open')
```
opens the shutter: the beam reaches the sample position.

```
move(stage_x, 500)
```
moves the sample stage 500 mm to the side: the plate is out of the beam, the camera sees the beam alone.

```
with nexusfile_open('open beam'): count(t=30)
```
counts for 30 s: a file titled *open beam*. The *Detector image* fills up evenly.

<span class="twin-nicos"></span>

### Step 5. The iron plate

```
move(stage_x, 0)
```
brings the plate back in the beam, at the center of the camera.

```
with nexusfile_open('iron plate'): count(t=30)
```
counts for 30 s, as long as the open beam, so that both have the same statistics. The *Detector image* now shows the
plate: a darker square, because the iron stops some of the neutrons.

```
list_filewriting_jobs()
```
lists your three files: dark, open beam, iron plate.

In [ ]:
q.choice("In step 4, why is the plate moved 500 mm away rather than taken off the instrument?",
         ["The camera sees only 14 mm × 14 mm: 500 mm aside, the plate is far out of the beam, and the stage brings it "
          "back exactly to the same place for step 5.",
          "Because 500 mm is the length of the plate.",
          "To protect the plate from the neutrons.",
          "Because the open beam must be measured through the plate."],
         answer=0, why={1: "The plate is 10 mm wide; 500 mm is the travel of the stage.",
                        2: "The plate is not damaged by the beam; the open beam is a measurement without it.",
                        3: "The open beam is measured without the sample: it is the reference."},
         explanation="Moving the stage is what an instrument scientist does: the sample leaves the 14 mm field of view, "
                     "and the motor brings it back to the same position to the micrometer.",
         hints=["How large is the field of view of the camera?"], title="Q2. The open beam")
q.choice("You forgot `move(shutter, 'open')` before the open beam. What will your open-beam file look like?",
         ["Like a second dark: almost no neutrons, so the transmission cannot be computed.",
          "Like the sample: the plate is still in the beam.",
          "Normal: the shutter only protects the camera.",
          "NICOS refuses to count."],
         answer=0, why={1: "The plate was moved away; with the shutter closed no neutron arrives at all.",
                        2: "A closed shutter stops the beam before the sample: nothing reaches the camera.",
                        3: "NICOS counts: it does not know what you meant to measure. You will see it in the logbook."},
         explanation="Part 2 reads the logbook from your files: a file with almost no neutrons and the shutter closed is "
                     "a dark, whatever its title. Measure the open beam again.",
         hints=["What does a closed shutter stop?"], title="Q3. A forgotten shutter")

<span class="twin-nicos"></span>

**When the three files are written**, come back here and run the whole notebook: menu *Run > Run All Cells*. The
notebook reads your files from part 1 on.

## 1. Your files

As at ESS, NICOS has written your files in the folder of your proposal, `data/<instrument>/<year>/<proposal>/raw`. The cell below goes to the folder of proposal 1234, this year, and lists the files it finds there. If there is no file yet, it stops with a message: measure first (part 0).

In [ ]:
from odin_paths import TWIN_RUN_ROOT          # root of the results of the twins
import datetime
PROPOSAL = "1234"                              # the proposal of this exercise (part 0)
FOLDER = f"{TWIN_RUN_ROOT}/data/odin/{datetime.date.today().year}/{PROPOSAL}/raw"

In [ ]:
import sys, os, json, glob, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import scipp as sc
import h5py
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import erfc
from odin_paths import TWIN_RUN_ROOT
import odin_reduction as red

import nb_tools
nb_tools.need_measurements(FOLDER, "odin")   # stops here, with a message, while there is no file
print("folder:", FOLDER)
print("files :", [os.path.basename(f) for f in red.data_files(FOLDER)])

## 2. The logbook: which file is what?

Each `nexusfile_open` block wrote **one file**, as at ESS (`odin_job_000001.hdf`, `odin_job_000002.hdf`, ...). Everything is in the file: its title, and the state of the instrument during the measurement (shutter, sample stage, pinhole, choppers). The table below is read from the files themselves.

The column `kind` is deduced from the state of the instrument, not from the title: shutter closed = dark; shutter open and stage far from 0 = open beam; shutter open and stage at 0 = sample. Check it against what you did.

In [ ]:
rows = red.measurements(FOLDER)                         # the logbook, read from the files
def show_table(rows):                                  # a list of dicts, printed as a table
    if not rows:
        print("no measurement in the logbook"); return
    cols = list(rows[0])
    width = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in cols}
    print("  ".join(c.ljust(width[c]) for c in cols))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(width[c]) for c in cols))

show_table(rows)

The cell below takes the **last** dark, open beam and sample of the logbook: those you have just measured. It also checks that the three were measured with the same chopper mode and pinhole: otherwise the ratio would mix two different beams and mean nothing (question Q1).

In [ ]:
def last(kind):
    names = [r["file"] for r in rows if r["kind"] == kind]
    return names[-1] if names else None

DARK, OPEN, SAMPLE = last("dark"), last("open beam"), last("sample")
print(f"dark: {DARK}   open beam: {OPEN}   sample: {SAMPLE}")
if OPEN is None or SAMPLE is None:
    print("An open beam and a sample measurement are needed: measure them, then run this notebook again.")
settings = {(r["choppers"], r["wfm_delta_m"], r["pinhole_m"]) for r in rows if r["file"] in (DARK, OPEN, SAMPLE)}
if len(settings) > 1:
    print("WARNING: these files were not measured with the same chopper mode and pinhole:", settings)
path = lambda name: f"{FOLDER}/{name}" if name else None

## 3. Inside a NeXus file

The files have the format of the real ODIN at ESS (NeXus, built on HDF5: a file organized like folders and
sub-folders). The detector (`timepix3`) does not record an image: it records **each neutron**, one by one, as an
*event*:
- `event_id`: the pixel hit;
- `event_time_offset`: the arrival time of the neutron, counted from the start of its source pulse (ESS sends
  14 pulses per second);
- `event_time_zero` / `event_index`: when each pulse happened and which events belong to it.

The cell below prints an extract of the open-beam file: the first source pulses, then the first five neutrons, exactly
as they are stored.

In [ ]:
import datetime
with h5py.File(path(OPEN), "r") as f:
    det = f["entry/instrument/timepix3"]
    t_zero = det["data/event_time_zero"][:3]       # ns since 1 January 1970 (UTC)
    index = det["data/event_index"][:3]
    ids = det["data/event_id"][:5]
    toa = det["data/event_time_offset"][:5]        # ns after the start of the pulse
    x_off, y_off = det["x_pixel_offset"][()], det["y_pixel_offset"][()]    # pixel centers [m]

print("source pulses")
print("  pulse   event_time_zero [ns]   = date and time (UTC)            event_index (first event of the pulse)")
for k, (tz, i0) in enumerate(zip(t_zero, index)):
    date = datetime.datetime.fromtimestamp(tz / 1e9, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")
    print(f"  {k:5d}   {tz:20d}   {date}   {i0:10d}")
print("\nfirst neutrons (all in pulse 0)")
print("  event   event_id (pixel)   -> x, y on the detector [mm]   event_time_offset [ns]   = [ms]")
for k, (pix, t) in enumerate(zip(ids, toa)):
    ix, iy = (pix - 1) % len(x_off), (pix - 1) // len(x_off)      # pixel number = 1 + ix + 4096 iy
    print(f"  {k:5d}   {pix:16d}   {x_off[ix] * 1e3:+6.2f}, {y_off[iy] * 1e3:+6.2f}"
          f"{'':15s}{t:16.0f}   {t / 1e6:7.3f}")

**Reading the extract.** Each neutron is just two numbers: a pixel number and an arrival time. The first neutron
hit pixel number `event_id` (the Timepix3 camera has 4096 × 4096 pixels, numbered row by row, which gives its position
on the 14 mm × 14 mm detector) `event_time_offset` nanoseconds after the start of pulse 0. The pulses are 71.4 ms apart (1/14 s):
compare the `event_time_zero` of pulses 0 and 1. `event_index` says where each pulse starts in the list of
events: the events of pulse 0 are those numbered from `event_index[0]` to `event_index[1]` − 1, and so on. An image,
a spectrum, a transmission: everything that follows is computed from these lists.

The file also contains the state of the instrument (choppers, pinhole, shutter, sample stage...) and the **proton
charge** of every pulse: the number of protons that hit the target, to which the number of neutrons produced is
proportional.

In [ ]:
with h5py.File(path(OPEN), "r") as f:
    e = f["entry"]
    print("title       :", e["title"][()].decode())
    print("sample      :", e["sample/name"][()].decode())
    print("start / end :", e["start_time"][()].decode(), "/", e["end_time"][()].decode())
    det = e["instrument/timepix3/data"]
    toa = det["event_time_offset"][()]
    unit = det["event_time_offset"].attrs.get("units", b"ns")
    unit = unit.decode() if isinstance(unit, bytes) else unit
    print("neutrons    :", len(toa), "events,", len(det["event_time_zero"]), "source pulses")
    charge = e["instrument/source/pulse_charge/value"][()]
    print(f"proton charge: {charge.sum():.0f} µC in total ({charge.mean():.1f} µC per pulse on average)")

scale = {"ns": 1e-6, "us": 1e-3, "ms": 1.0, "s": 1e3}.get(unit, 1e-6)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(toa * scale, bins=300, histtype="step", color="tab:blue")
ax.set_xlabel("arrival time after the source pulse [ms]"); ax.set_ylabel("neutrons per bin")
ax.set_title(f"{OPEN} (open beam): arrival times of the neutrons"); ax.grid(alpha=0.3)
plt.show()

The file shown is the open beam: the beam itself, without the sample.

**Reading the histogram.** A slow neutron (long wavelength) takes longer to fly the 60 m from the source to the
detector: the arrival time *is* a measurement of the wavelength, $\lambda = h\,t / (m_n L)$ (de Broglie). In WFM mode
(*wavelength frame multiplication*) the choppers cut the long ESS pulse into several short sub-pulses, the **frames**:
you see them as separate humps, each one covering a band of wavelengths. Converting time to wavelength therefore needs
the chopper settings, which are in the file: this is what the ESS software does in the next step.

In [ ]:
q.choice("What does the camera record for each neutron?",
         ["Which pixel it hit and when it arrived after the source pulse.",
          "Its wavelength, measured directly.",
          "An image, one per second.",
          "Its energy, measured by the pixel."],
         answer=0, why={1: "The wavelength is computed later, from the arrival time (part 4).",
                        2: "The file holds events, one per neutron; images are made from them.",
                        3: "A pixel only says that a neutron arrived, and when."},
         explanation="Two numbers per neutron: event_id (the pixel) and event_time_offset (the arrival time). Images, "
                     "spectra and transmissions are all computed from these lists.",
         hints=["Look at the two columns printed by the cell above."], title="Q4. What is in the file?")

## 4. Raw counts: open beam and sample

Before any correction, compare the number of neutrons detected with and without the sample, as a function of the
wavelength (left) and on the detector (right). The wavelength $\lambda$ of every neutron is computed by the official
ESS workflow (`essimaging`) from its arrival time and the chopper settings. The spectrum is summed over the region of
interest (ROI), a square of 6 mm × 6 mm at the center of the detector, where the iron plate is.

In [ ]:
LAM = sc.linspace("wavelength", 1.0, 9.0, 201, unit="angstrom")    # 200 bins of 0.04 Å
ROI = (-3, 3, -3, 3)                                                 # x_min, x_max, y_min, y_max [mm]
raw = {k: red.histograms(path(n), LAM, ROI) for k, n in (("open beam", OPEN), ("sample", SAMPLE))}

lam = sc.midpoints(LAM).values
fig, ax = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw=dict(width_ratios=[1.4, 1, 1]))
for (k, h), c in zip(raw.items(), ("tab:gray", "tab:red")):
    ax[0].step(lam, h["spectrum"].values, where="mid", color=c, label=f"{k} ({h['duration']:.0f} s)")
ax[0].set_xlabel("wavelength λ [Å]"); ax[0].set_ylabel("neutrons per 0.04 Å bin (ROI)")
ax[0].set_title("spectra in the ROI"); ax[0].legend(); ax[0].grid(alpha=0.3)
for a, (k, h) in zip(ax[1:], raw.items()):
    im = h["image"]
    x, y = im.coords["x"].values, im.coords["y"].values
    m = a.imshow(im.values, origin="lower", extent=[x[0], x[-1], y[0], y[-1]], cmap="viridis")
    a.add_patch(plt.Rectangle((ROI[0], ROI[2]), ROI[1] - ROI[0], ROI[3] - ROI[2], fill=False, ec="w", ls="--"))
    a.set_title(f"{k}: neutrons per pixel"); a.set_xlabel("x [mm]"); a.set_ylabel("y [mm]")
    plt.colorbar(m, ax=a, shrink=0.8)
plt.tight_layout(); plt.show()

**What to see.** The open-beam spectrum is the shape of the beam: it is not flat (the moderator emits more neutrons
around 2–4 Å) and shows the frames of the choppers. The sample spectrum is lower (iron absorbs and scatters
neutrons) and has **steps**. On the images, the plate (10 mm × 10 mm) is the darker square.

The raw spectra cannot be compared directly: the beam shape is in both, and the two counts may not have received the
same number of protons. The reduction removes all this.

In [ ]:
q.choice("The sample spectrum is lower than the open-beam spectrum. Why can the transmission not simply be the ratio "
         "of these two raw spectra?",
         ["Both contain the background of the dark, and the source may not have sent the same number of protons "
          "during the two counts.",
          "Because the two spectra have different wavelengths.",
          "It can: the raw ratio is the transmission.",
          "Because the plate is not in the ROI."],
         answer=0, why={1: "Both are computed on the same wavelength bins.",
                        2: "The background and the power of the source would bias it (part 5 corrects both).",
                        3: "The ROI (white square) is on the plate."},
         explanation="The dark is subtracted from both, and each count is divided by its proton charge (the number of "
                     "neutrons produced is proportional to it): this is the reduction of part 5.",
         hints=["What else than the plate is in both measurements?"], title="Q5. Raw counts")

## 5. Reduction: the transmission

The **transmission** is the fraction of neutrons that cross the sample without being absorbed or deflected:

$$T(\lambda) = \frac{\big(S - D\,t_S/t_D\big)\,/\,Q_S}{\big(O - D\,t_O/t_D\big)\,/\,Q_O}$$

- $S$, $O$, $D$: counts of the sample, open beam and dark runs; $t$: their durations;
- the dark is subtracted from both, scaled to the duration of each measurement;
- each measurement is divided by its proton charge $Q$, because the power of the source varies;
- the ratio removes the beam shape: $T = 1$ where nothing is in the beam, $T = 0$ for an opaque sample.

The error bars come from the counting statistics (Poisson: $\sqrt{N}$ on $N$ counts), propagated through the formula.
`red.reduce` does all this with the official ESS code (about 1 min).

In [ ]:
r = red.reduce(path(SAMPLE), path(OPEN), dark=path(DARK), lam=LAM, roi=ROI)
print({k: f"{v['duration']:.0f} s, {v['charge_uC']:.0f} µC" for k, v in r["normalization"].items()},
      "(E: sample, O: open beam, N: dark)")
T, dT = r["spectrum"].values, np.sqrt(r["spectrum"].variances)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw=dict(width_ratios=[1.6, 1]))
ax[0].errorbar(lam, T, dT, fmt=".", ms=3, lw=0.6, color="k")
ax[0].set_xlabel("wavelength λ [Å]"); ax[0].set_ylabel("transmission T"); ax[0].set_ylim(0, 1.05)
ax[0].set_title("transmission in the ROI"); ax[0].grid(alpha=0.3)
im = r["image"]; x, y = im.coords["x"].values, im.coords["y"].values
m = ax[1].imshow(im.values, origin="lower", extent=[x[0], x[-1], y[0], y[-1]], cmap="gray", vmin=0, vmax=1.2)
ax[1].set_title("transmission image (all wavelengths)"); ax[1].set_xlabel("x [mm]"); ax[1].set_ylabel("y [mm]")
plt.colorbar(m, ax=ax[1], shrink=0.8, label="T")
plt.tight_layout(); plt.show()

**Reading the figure.** Around the plate the image is close to $T = 1$ (nothing in the beam); on the plate
$T \approx 0.6$. The spectrum on the left is the fingerprint of the material: $T$ drops slowly in some places
and **jumps up** suddenly at a few wavelengths. These jumps are the **Bragg edges**.

**Why a jump?** Iron is a crystal: its atoms sit on planes a distance $d$ apart. Neutrons are deflected (Bragg
diffraction) by the planes when $\lambda = 2d\sin\theta$, $\theta$ being the angle between the neutron and the planes.
Since $\sin\theta \le 1$, a family of planes can deflect only neutrons with $\lambda \le 2d$. Just above $\lambda = 2d$
these planes stop removing neutrons from the beam, so the transmission jumps up. The position of the edge measures
$d$ directly: $d = \lambda_\text{edge}/2$.

In [ ]:
q.choice("On the plate, T ≈ 0.6 at many wavelengths. What does it mean?",
         ["60 % of the neutrons of that wavelength cross the 5 mm of iron without being stopped or deflected.",
          "60 % of the neutrons are absorbed.",
          "The plate is 60 % iron.",
          "The camera detects 60 % of the neutrons."],
         answer=0, why={1: "T is the fraction that comes through; 40 % are stopped or deflected.",
                        2: "The plate is pure iron; T depends on its thickness and on λ.",
                        3: "The efficiency of the camera cancels in the ratio sample / open beam."},
         explanation="T = 1 means nothing in the beam, T = 0 an opaque sample. The jumps of T with λ are the Bragg "
                     "edges: part 6 models them.", hints=["T is a ratio: of what to what?"], title="Q6. Transmission")

## 6. Your turn: model the (110) Bragg edge

α-iron is body-centered cubic, $a = 2.8665$ Å. The planes that can diffract are (110), (200), (211)... with spacings
$d_{hkl} = a/\sqrt{h^2+k^2+l^2}$, and each family gives an edge at $\lambda = 2d_{hkl}$: by Bragg's law,
$\lambda = 2d\sin\theta$, a wavelength longer than $2d$ cannot be diffracted by these planes at any angle, so the
neutrons go through and the transmission jumps up.

You develop the analysis with the **model builder** below; nothing has to be typed.

1. **Choose the shape of the edge.** Start with the simplest, a sharp step, and look at the residuals.
2. **Choose the range** of the data: the edge, with some baseline on each side.
3. **Start values**: move the sliders until the dashed curve is close to the data, then press **Fit**.

Try the three shapes, and different ranges. For each fit the builder gives $\lambda_\text{edge}$ with its uncertainty,
the residuals (data − model, in units of the error bar) and the **reduced χ²**: about 1 when the model describes the
data within their error bars, larger when it misses something (the residuals then show a pattern instead of noise).
Every fit is kept in a table, so you can compare them. Under *See the code*: the Python lines your choices wrote.

In [ ]:
import model_builder as mb
expected = 2 * 2.8665 / np.sqrt(2)          # (110) edge: λ = 2 d_110 = a √2
edge_builder = mb.EdgeBuilder(lam, T, dT, expected=expected, name="(110)")

In [ ]:
q.choice("Which shape describes the (110) edge best, and why?",
         ["The smoothed step on a sloping baseline: reduced χ² close to 1 and residuals without a pattern.",
          "The sharp step: it is the ideal edge of a perfect crystal.",
          "The step smoothed by the resolution, on a flat baseline: the baseline cannot vary over 0.7 Å.",
          "Any of them: they all give the edge within 0.01 Å."],
         answer=0, why={1: "The measured edge is not sharp: the wavelength resolution of the instrument smooths it. "
                           "Look at the residuals of the sharp step near the edge.",
                        2: "Fit both and compare: the χ² of the flat baseline stays above 1, and the residuals rise "
                           "and fall on each side of the edge. The other planes and the absorption make the baseline "
                           "vary with λ.",
                        3: "The positions differ by more than their uncertainties, and a sharp step gives no "
                           "uncertainty at all. The model matters."},
         explanation="The resolution smooths the edge (an error function) and the baseline varies slowly with λ: "
                     "only the third model leaves residuals that look like noise (χ²/n ≈ 1).",
         hints=["Compare the reduced χ² of your fits in the table.", "Look at the residuals: noise, or a pattern?"],
         title="Choice of the model")

**Check your edge.** The twin knows the true sample. If it is wrong, change the model, the range or the start values, fit again and run this cell again.

In [ ]:
ex.check_edge(edge_builder.lam_edge, edge_builder.dlam_edge)

In [ ]:
q.solution("part 6", explanation="A step smoothed by a Gaussian resolution of width $w$ is an error function, "
           "$T = A + C(\\lambda-\\lambda_e) + \\frac{B}{2}\\,\\mathrm{erfc}(-(\\lambda-\\lambda_e)/(\\sqrt2\\,w))$, on a "
           "sloping baseline: the third shape, in a range of about ±0.35 Å around the expected edge.")

**From the edge to the lattice.** At the edge $\lambda = 2d_{110}$, and for a cubic lattice $d_{110} = a/\sqrt2$.

In [ ]:
q.number("With your edge, what is the lattice parameter $a$ of the plate, in Å?", value=2.8665, tol=0.006, unit="Å",
         mistakes=[(4.054, "the wavelength of the edge itself, not a"), (2.027, "d₁₁₀ = λ/2; a is √2 times larger"),
                   (1.433, "λ/(2√2): one factor √2 too many")],
         hints=["λ = 2 d₁₁₀, so d₁₁₀ = λ/2.", "a = √2 · d₁₁₀ = λ/√2."],
         explanation="a = λ_edge/√2 ≈ 4.06/1.414 = 2.87 Å; its uncertainty is that of λ_edge divided by √2.",
         title="Lattice parameter")

**Your verdict for Fyrø.**

In [ ]:
q.choice("γ-iron (face-centered cubic, a = 3.59 Å) would have its (111) edge at 4.15 Å and a (200) edge at 3.59 Å. "
         "Your plate has its last edge at about 4.06 Å and a lattice parameter of about 2.87 Å. What do you write?",
         ["Ferritic steel (α-iron), a = 2.87 Å: as ordered.", "Austenitic steel (γ-iron): send it back.",
          "Cannot be decided from the transmission.", "A mixture of both, half and half."],
         answer=0, why={1: "γ-iron would put its last edge at 4.15 Å, 0.09 Å away from yours: about a hundred times "
                           "your uncertainty.",
                        2: "The positions of the edges are set by the crystal lattice: they decide.",
                        3: "A mixture would show the edges of both lattices (4.15 and 3.59 Å as well): your spectrum "
                           "has only those of a body-centered cubic lattice."},
         explanation="The edges are those of a body-centered cubic lattice with a = 2.87 Å: ferritic steel. The "
                     "difference to γ-iron is far larger than your uncertainty.", hints=['Compare your last edge (about 4.06 Å) with that of γ-iron (4.15 Å): how many uncertainties apart?', 'Your lattice parameter is about 2.87 Å; γ-iron has 3.59 Å.'],
         title="Verdict")

## 7. Developing the method: a systematic error

Your edge is probably a few thousandths of an ångström away from the true value, more than its uncertainty. A counting
fluctuation would shrink if you counted longer; this does not. Find where it comes from (nothing to type: use the
builders and run the cells):

1. **The range.** In the builder of part 6, fit with a 2× wider and a 2× narrower range. Does $\lambda_\text{edge}$
   move by more than its uncertainty?
2. **The time → wavelength conversion.** The ESS reduction converts the arrival time into λ with a lookup table. The
   default table (`"simulation"`) uses a realistic ESS pulse; the other (`"analytical"`) assumes a simple pulse. The
   cell below reduces your files again with the other table and opens a second builder: fit the same model and
   compare the two edges with the truth (the check cell after it). Which table is closer, and why (the shape of the
   ESS pulse)?
3. **Counting statistics.** Measure the sample and the open beam 4 times longer in NICOS, run the notebook again: by
   how much do the uncertainty and the distance to the truth change?

In [ ]:
r_analytical = red.reduce(path(SAMPLE), path(OPEN), dark=path(DARK), lam=LAM, roi=ROI, wavelength_from="analytical")
T_an, dT_an = r_analytical["spectrum"].values, np.sqrt(r_analytical["spectrum"].variances)
edge_builder_analytical = mb.EdgeBuilder(lam, T_an, dT_an, expected=expected, name="(110), analytical table")

In [ ]:
print("default table (simulation):"); ex.check_edge(edge_builder.lam_edge, edge_builder.dlam_edge)
print("analytical table:");           ex.check_edge(edge_builder_analytical.lam_edge, edge_builder_analytical.dlam_edge)

Write your conclusion below: what limits the accuracy of a Bragg-edge measurement, and how would you calibrate it at a
real instrument? Then check your understanding.

## My answer

(your conclusion)

In [ ]:
q.choice("You count 4 times longer. What happens to the uncertainty of the edge and to the distance between your edge "
         "and the truth?", ["The uncertainty is halved; the distance to the truth stays about the same.",
                            "Both are halved.", "Both stay the same.", "The uncertainty is divided by 4."],
         answer=0, why={1: "The distance to the truth is a systematic error: it does not depend on the number of "
                           "neutrons.",
                        2: "More neutrons always shrink the statistical uncertainty: as 1/√N.",
                        3: "The uncertainty goes as 1/√N: √4 = 2, not 4."},
         explanation="Statistical errors shrink as 1/√N (halved for 4 times more counts); a systematic error does "
                     "not. When the distance to the truth is several times the uncertainty, the error is systematic.",
         hints=['Which of the two depends on the number of neutrons counted?', 'A statistical uncertainty goes as 1/√N; an offset of the wavelength scale does not depend on N.'],
         title="Statistical or systematic?")
q.choice("How is such a systematic error corrected at a real instrument?",
         ["By measuring a reference sample whose plane spacings are known, and correcting the wavelength scale.",
          "By counting longer.", "By using a wider fit range.", "It cannot be corrected."],
         answer=0, why={1: "Counting longer only shrinks the statistical error.",
                        2: "The range changes the fit a little but does not remove an error of the wavelength scale.",
                        3: "It can: the wavelength scale is calibrated."},
         explanation="The error comes from the conversion of the arrival time into a wavelength (flight distances, "
                     "chopper timings, shape of the ESS pulse). Instruments calibrate it with a reference powder of "
                     "known lattice parameter.", hints=['The offset comes from the wavelength scale. How would you check a scale?', 'Measure something whose answer you know already.'],
         title="Calibration")
q.report()

## Submit your work

When you are done: save this notebook (*Save*), then submit. The file for your teacher is made from your answers, your measurements and your notebook.

In [ ]:
import submit
_ = submit.Submit("odin/exercises/iron_bragg_edges", "msc", quiz=q)